## Levantar el entorno

Descarga la data (si falta) y levanta el cluster de Spark (master + 2 workers) y Jupyter con Docker

- Kernel picker → "Existing Jupyter Server..." → `http://localhost:8888`
- UI de Spark: `http://localhost:8080` (master), `http://localhost:4040` (job activo)

In [ ]:
import json, os, subprocess, sys, time, urllib.request
from pathlib import Path

proyecto = Path.cwd()
while not (proyecto / "docker-compose.yml").exists() and proyecto != proyecto.parent:
    proyecto = proyecto.parent
print("Proyecto:", proyecto)

### 1. Docker disponible

In [ ]:
r = subprocess.run(["docker", "compose", "version"], capture_output=True, text=True)
print(r.stdout or "Docker no responde: abre Docker Desktop y vuelve a ejecutar")

### 2. Descarga de la data (OECE, 2023-2025)

Se omite si `data/raw` ya tiene los JSON

In [ ]:
raw = proyecto / "data" / "raw"
if not any(raw.glob("*/*.json")):
    subprocess.run([sys.executable, str(proyecto / "data" / "download.py")], check=True)
print(f"{len(list(raw.glob('*/*.json')))} archivos en data/raw")

### 3. Levantar contenedores

In [ ]:
env = {**os.environ}
if hasattr(os, "getuid"):
    env.setdefault("HOST_UID", str(os.getuid()))
r = subprocess.run(["docker", "compose", "up", "-d"], cwd=proyecto, env=env, capture_output=True, text=True)
print(r.stdout, r.stderr)

### 4. Esperar a que respondan los servicios

In [ ]:
def esperar(url, intentos=60):
    for _ in range(intentos):
        try:
            urllib.request.urlopen(url, timeout=3)
            return True
        except Exception:
            time.sleep(5)
    return False

for nombre, url in {"Spark master": "http://localhost:8080", "Jupyter": "http://localhost:8888/api"}.items():
    print(f"{nombre:13s}: {'OK' if esperar(url) else 'NO RESPONDE'}  {url}")

In [ ]:
estado = json.loads(urllib.request.urlopen("http://localhost:8080/json/").read())
print("workers vivos:", sum(w["state"] == "ALIVE" for w in estado["workers"]), "| cores:", estado["cores"])
print(subprocess.run(["docker", "compose", "ps"], cwd=proyecto, capture_output=True, text=True).stdout)